# signVLM training (Google Colab)

This notebook follows the **SignVLM** stack in this repository (`main.py`, `model.py`, `video_dataset/`): CLIP ViT backbone + temporal decoder + classifier, **AdamW**, **CosineAnnealingLR**, **CrossEntropyLoss**, **AMP fp16**, and list files in the form `path<TAB>label` (see `SIGNVLM_TRAINING_NOTES.md` and `docs/signVLMPipeline.md`).

**Cell 1 — Setup:** installs PyAV (`av`), **scikit-learn** (confusion matrix + F1), **pillow**, and prints PyTorch/CUDA.

**Cell 2** mounts Drive and sets `BASE`, checkpoints, list dir, and default **CLIP weights** on Drive (`DRIVE_REPO_ROOT/CLIP_weights/...`). **Cell 2b** loads training **Python code** from **GitHub** (`SIGNVLM_GIT_URL` + `SIGNVLM_GIT_BRANCH`) into `/content/...`, or uses a full repo already on Drive under `DRIVE_REPO_ROOT` if `main.py` exists there. **Data** stays on Drive; **Cell 4** uses **`local_data_cache`** to copy each sample to `/content` on first read (see below).

In [1]:
# Cell 1: Setup and installations
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["av", "pillow", "scikit-learn"])
print("OK: av (PyAV), pillow, scikit-learn")

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

OK: av (PyAV), pillow, scikit-learn
torch: 2.10.0+cu128 cuda: True


## Cell 2: Mount Drive and paths

On **Colab**, `BASE` should live under **`/content/drive/MyDrive/...`** so **checkpoints** (`signVLM_checkpoints/`) and **list files** (`signVLM_lists/`) persist on **Google Drive**, not ephemeral `/content`. With default `BASE` (`.../MyDrive/FYP_DATA`), checkpoints are **`FYP_DATA/signVLM_checkpoints/checkpoint-*.pth`**. **Resume** (Cell 6 + `checkpoint.py`) picks the file with the **largest** step number in the name (e.g. `checkpoint-7500.pth` if that is the highest present).

**Overrides:** set `SIGNVLM_DRIVE_BASE` (full path to your dataset root), `SIGNVLM_BACKBONE_PATH` (ViT-L `.pt`), or optional `SIGNVLM_TRAIN_SUBDIR` / `VAL` / `TEST` / `UNSEEN` / `REPO_SUBDIR`. Off Colab, use `SIGNVLM_BASE` or rely on `./signvlm_data`.

**`WORK_ROOT`** defaults to `/content`. **`CODE_ROOT`** is set in **Cell 2b** (git clone path or `DRIVE_REPO_ROOT`). Default **ViT weights** use `DRIVE_REPO_ROOT/CLIP_weights/...` on Drive (override with `SIGNVLM_BACKBONE_PATH`).

Expected layout under `BASE`:

- `train_data/<class_name>/*.mp4` (or frames under `<class>/<stem>/` if using `frames_available=1`)
- `validation_data/...`
- `test_data/...`
- `Unseen_data/...` (optional; Cell 4 writes an empty `unseen.tsv` if missing)

**Cell 2b (training code):** set **`SIGNVLM_GIT_URL`** to your public or token-based clone URL (e.g. run `import os; os.environ['SIGNVLM_GIT_URL'] = 'https://github.com/org/repo.git'` in a cell before 2b). Optional: **`SIGNVLM_GIT_BRANCH`**, **`SIGNVLM_GIT_DEPTH`**, **`SIGNVLM_GIT_DIR`**, **`SIGNVLM_GIT_RECLONE=1`**. If unset and **`DRIVE_REPO_ROOT/main.py`** exists, the Drive copy is used. **Private repos:** token-in-URL or Colab **Secrets** (error text in 2b). **Weights** default to **`DRIVE_REPO_ROOT/CLIP_weights/...`** in Cell 2, not the git tree.

**Data I/O (Colab):** keep media on Drive; **Cell 4** sets `local_data_cache` so each file is copied to `/content` on first read (no bulk staging).


In [ ]:
# Cell 2: Mount Google Drive and directory paths
# Checkpoints go to CHECKPOINT_DIR; on Colab keep BASE under Drive so artifacts survive session loss.
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_ROOT = "/content"
CODE_ROOT = f"{WORK_ROOT}/signvlm_bundle"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MY = "/content/drive/MyDrive"
else:
    DRIVE_MY = os.environ.get("SIGNVLM_DRIVE_MY", "")

BASE = os.environ.get("SIGNVLM_DRIVE_BASE", "").strip()
if not BASE:
    if IN_COLAB and DRIVE_MY:
        BASE = f"{DRIVE_MY}/FYP_DATA"
    else:
        BASE = os.environ.get("SIGNVLM_BASE", os.path.join(os.getcwd(), "signvlm_data"))

TRAIN_SUBDIR = os.environ.get("SIGNVLM_TRAIN_SUBDIR", "train_data")
VAL_SUBDIR = os.environ.get("SIGNVLM_VAL_SUBDIR", "validation_data")
TEST_SUBDIR = os.environ.get("SIGNVLM_TEST_SUBDIR", "test_data")
UNSEEN_SUBDIR = os.environ.get("SIGNVLM_UNSEEN_SUBDIR", "unseen_data")

TRAIN_DIR = f"{BASE}/{TRAIN_SUBDIR}"
VAL_DIR = f"{BASE}/{VAL_SUBDIR}"
TEST_DIR = f"{BASE}/{TEST_SUBDIR}"
UNSEEN_DIR = f"{BASE}/{UNSEEN_SUBDIR}"

REPO_SUBDIR = os.environ.get("SIGNVLM_REPO_SUBDIR", "Urdu-Sign-Language-Recognition-using-SignVLM")
DRIVE_REPO_ROOT = f"{BASE}/{REPO_SUBDIR}"
REPO_ROOT = DRIVE_REPO_ROOT

BACKBONE_PATH = os.environ.get("SIGNVLM_BACKBONE_PATH", "").strip()
if not BACKBONE_PATH:
    BACKBONE_PATH = f"{DRIVE_REPO_ROOT}/CLIP_weights/ViT-L/ViT-L/ViT-L-14.pt"

CHECKPOINT_DIR = f"{BASE}/signVLM_checkpoints"
LIST_DIR = f"{BASE}/signVLM_lists"

if IN_COLAB and not CHECKPOINT_DIR.startswith("/content/drive/MyDrive"):
    raise RuntimeError(
        "CHECKPOINT_DIR must be under mounted Google Drive (/content/drive/MyDrive/...). "
        "Set SIGNVLM_DRIVE_BASE to a path under MyDrive, not /content alone."
    )

if not os.path.isfile(BACKBONE_PATH):
    raise FileNotFoundError(
        f"CLIP backbone not found: {BACKBONE_PATH}\n"
        "Set SIGNVLM_BACKBONE_PATH or place ViT-L-14.pt under DRIVE_REPO_ROOT/CLIP_weights/... on Drive."
    )

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LIST_DIR, exist_ok=True)
print(
    f"IN_COLAB={IN_COLAB} WORK_ROOT={WORK_ROOT} CODE_ROOT={CODE_ROOT} BASE={BASE} "
    f"DRIVE_REPO_ROOT={DRIVE_REPO_ROOT} REPO_ROOT={REPO_ROOT} (overridden in 2b if using git) "
    f"BACKBONE_PATH={BACKBONE_PATH} CHECKPOINT_DIR={CHECKPOINT_DIR} LIST_DIR={LIST_DIR}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True WORK_ROOT=/content CODE_ROOT=/content/signvlm_bundle BASE=/content/drive/MyDrive/FYP_DATA DRIVE_REPO_ROOT=/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM REPO_ROOT=/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM (overridden in 2b if using git) BACKBONE_PATH=/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM/CLIP_weights/ViT-L/ViT-L/ViT-L-14.pt CHECKPOINT_DIR=/content/drive/MyDrive/FYP_DATA/signVLM_checkpoints LIST_DIR=/content/drive/MyDrive/FYP_DATA/signVLM_lists


In [3]:
# Cell 2b: Training code from GitHub (preferred) or a full repo copy on Drive
import os
import shutil
import subprocess

WORK_ROOT = globals().get("WORK_ROOT", "/content")
CODE_ROOT = globals().get("CODE_ROOT", f"{WORK_ROOT}/signvlm_bundle")
DRIVE_REPO_ROOT = globals().get("DRIVE_REPO_ROOT", "")
REPO_ROOT = globals().get("REPO_ROOT", DRIVE_REPO_ROOT)

# Defaults point at the requested repo/branch; env vars can still override them.
DEFAULT_GIT_URL = "https://github.com/ZainnB/Urdu-Sign-Language-Recognition-using-SignVLM.git"
DEFAULT_GIT_BRANCH = "training_thru_tensors"
GIT_URL = os.environ.get("SIGNVLM_GIT_URL", DEFAULT_GIT_URL).strip()
GIT_BRANCH = (os.environ.get("SIGNVLM_GIT_BRANCH", DEFAULT_GIT_BRANCH) or DEFAULT_GIT_BRANCH).strip()
try:
    GIT_DEPTH = int(os.environ.get("SIGNVLM_GIT_DEPTH", "1") or "1")
except ValueError:
    GIT_DEPTH = 1
GIT_DIR = os.environ.get("SIGNVLM_GIT_DIR", "").strip() or os.path.join(
    WORK_ROOT, "Urdu-Sign-Language-Recognition-using-SignVLM"
)
RECLONE = os.environ.get("SIGNVLM_GIT_RECLONE", "").strip().lower() in ("1", "true", "yes")


def _has_main(path: str) -> bool:
    return bool(path) and os.path.isfile(os.path.join(path, "main.py"))


if GIT_URL:
    if RECLONE and os.path.isdir(GIT_DIR):
        shutil.rmtree(GIT_DIR, ignore_errors=True)
    if os.path.isdir(os.path.join(GIT_DIR, ".git")):
        subprocess.run(
            ["git", "-C", GIT_DIR, "fetch", "--all", "--prune"],
            check=False,
        )
        subprocess.run(["git", "-C", GIT_DIR, "checkout", GIT_BRANCH], check=True)
        subprocess.run(
            ["git", "-C", GIT_DIR, "pull", "--ff-only"],
            check=False,
        )
    else:
        if os.path.exists(GIT_DIR):
            shutil.rmtree(GIT_DIR, ignore_errors=True)
        parent = os.path.dirname(GIT_DIR)
        if parent:
            os.makedirs(parent, exist_ok=True)
        cmd = ["git", "clone"]
        if GIT_DEPTH > 0:
            cmd.extend(["--depth", str(GIT_DEPTH)])
        cmd.extend(["-b", GIT_BRANCH, GIT_URL, GIT_DIR])
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
    REPO_ROOT = GIT_DIR
    CODE_ROOT = GIT_DIR
    print("Using SignVLM from git:", CODE_ROOT, "branch:", GIT_BRANCH)
elif _has_main(DRIVE_REPO_ROOT):
    REPO_ROOT = DRIVE_REPO_ROOT
    CODE_ROOT = DRIVE_REPO_ROOT
    print("Using SignVLM from Drive (DRIVE_REPO_ROOT):", CODE_ROOT)
else:
    _msg = (
        "No training code found. Either set environment variable SIGNVLM_GIT_URL to your "
        "GitHub clone URL and re-run this cell, or place a full repo with main.py under:\n  "
        + DRIVE_REPO_ROOT
        + "\nFor private repos, use: https://<token>@github.com/<org>/<repo>.git (Colab: store token in User secrets).\n"
        "Optional: SIGNVLM_GIT_BRANCH (default main), SIGNVLM_GIT_DEPTH (default 1), "
        "SIGNVLM_GIT_DIR (clone path), SIGNVLM_GIT_RECLONE=1 to delete and re-clone."
    )
    raise FileNotFoundError(_msg)


Using SignVLM from git: /content/Urdu-Sign-Language-Recognition-using-SignVLM branch: training_thru_tensors


## Cell 3: Imports

Run **Cell 2** (Drive paths + `DRIVE_REPO_ROOT` + `BACKBONE_PATH` on Drive), then **Cell 2b** (git clone or Drive `main.py`). After 2b, `CODE_ROOT` is the **git** checkout or your Drive repo. Initializes **single-GPU distributed** like `main.py` (FileStore + `gloo`).

In [4]:
# Cell 3: Imports and single-process distributed init (matches main.py)
# Run Cell 2, then Cell 2b — training code lives in CODE_ROOT (Cell 2 / 2b).
import os
import sys
import tempfile
import atexit

CODE_ROOT = globals().get("CODE_ROOT", "/content/signvlm_bundle")

if not os.path.isfile(os.path.join(CODE_ROOT, "main.py")):
    raise FileNotFoundError(
        "SignVLM sources not found. Run Cell 2b (set SIGNVLM_GIT_URL or place main.py under DRIVE_REPO_ROOT). Expected: "
        + os.path.join(CODE_ROOT, "main.py")
    )

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)

import torch
import torch.distributed as dist

if not dist.is_initialized():
    _store_path = os.path.join(tempfile.gettempdir(), f"signvlm_store_{os.getpid()}")
    _store = dist.FileStore(_store_path, 1)
    dist.init_process_group(backend="gloo", store=_store, rank=0, world_size=1)
    atexit.register(lambda p=_store_path: os.remove(p) if os.path.exists(p) else None)

import checkpoint
import video_dataset
from model import EVLTransformer
from weight_loaders import weight_loader_fn_dict
from vision_transformer import vit_presets

print("Code (sys.path):", CODE_ROOT)
print("REPO_ROOT (Cell 2 paths / CLIP weights):", globals().get("REPO_ROOT", "<run Cell 2 for Drive paths>"))
print("CUDA:", torch.cuda.is_available())


Code (sys.path): /content/Urdu-Sign-Language-Recognition-using-SignVLM
REPO_ROOT (Cell 2 paths / CLIP weights): /content/Urdu-Sign-Language-Recognition-using-SignVLM
CUDA: True


### Confusion matrix & F1 helpers

`collect_predictions` matches `main.evaluate` (batch size 1, multi-view softmax mean). `print_confusion_and_f1` prints macro/weighted/micro F1, `classification_report`, and the confusion matrix (or saves `.npy` when `num_classes` is large).

In [5]:
# Metrics (same inference as main.evaluate)
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score


def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data, labels in loader:
            data = data.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)
            assert data.size(0) == 1
            if data.ndim == 6:
                data = data[0]
            logits = model(data)
            scores = logits.softmax(dim=-1).mean(dim=0)
            y_pred.append(int(scores.argmax().cpu()))
            y_true.append(int(labels.view(-1)[0].cpu()))
    return np.array(y_true), np.array(y_pred)


def print_confusion_and_f1(
    y_true,
    y_pred,
    num_classes,
    class_names,
    title,
    save_dir=None,
    max_print_classes=35,
    print_report=True,
    print_matrix=True,
):
    labels_idx = np.arange(num_classes)
    cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
    names = [class_names[i] if i < len(class_names) else str(i) for i in range(num_classes)]
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"F1 (macro):     {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 (weighted):  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 (micro):     {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")

    if print_report:
        print("\nClassification report:")
        print(
            classification_report(
                y_true, y_pred, labels=labels_idx, target_names=names, zero_division=0, digits=4
            )
        )

    if save_dir:
        import os
        safe = title.lower().replace(" ", "_").replace("/", "_")
        p = os.path.join(save_dir, f"{safe}_confusion_matrix.npy")
        np.save(p, cm)
        print(f"Saved full matrix: {p}")

    if not print_matrix:
        print(f"\nConfusion matrix shape {cm.shape} (matrix printing disabled).")
        return

    if num_classes <= max_print_classes:
        print("\nConfusion matrix (rows=true class, cols=predicted class):")
        print(cm)
    else:
        print(f"\nConfusion matrix shape {cm.shape} (too large for full print).")
        h = min(12, cm.shape[0])
        print(f"Top-left {h}x{h} block:\n{cm[:h, :h]}")

## Cell 4: Data preparation — TSV lists + loaders

- Builds a **class name → id** map from sorted folder names under `train_data` (ids `0 .. num_classes-1`).
- Writes `train.tsv`, `val.tsv`, `test.tsv`, and `unseen.tsv` (empty if `Unseen_data` is missing) with lines `relative_path<TAB>label`, paths relative to each split root (same convention as `prepare_psl_splits.py` when `data_root` is the split folder).
- **Cell 4 speed toggle:** set env `SIGNVLM_REUSE_LISTS=1` (default behavior) to reuse existing non-empty `*.tsv` files instead of rescanning directories every rerun. Set `SIGNVLM_REUSE_LISTS=0` to force full rebuild when dataset contents changed.
- **Videos:** `frames_available=0` (PyAV). For **pre-extracted frames**, set `FRAMES_AVAILABLE = 1` and use the repo’s `tools/extract_frames_signvlm.py` first; lists must use virtual `.mp4` paths as in `SIGNVLM_TRAINING_NOTES.md`.
- Adjust **`NUM_CLASSES`** if you use a fixed label map file instead (must match folder names / ids).

### DataLoader / Colab notes

- **`local_data_cache` (default on Colab):** `video_dataset` copies each list item from the split root into `/content/signvlm_data_cache/...` on **first** access and reuses the local copy for **all later** steps. Set `LOCAL_DATA_CACHE = ""` in the code cell to read from Drive only (slower).
- **Hardcoded training speed profile (Cell 4 + 6):** edit `TRAIN_SPEED_PROFILE` in the code cell.
  - `"fast_stable"` = safer defaults for fewer crashes.
  - `"max_speed"` = higher throughput (more workers / bigger batch / single split), may need more RAM/VRAM.
  - You can still fine-tune by directly editing `PROFILE_BATCH_SIZE`, `PROFILE_NUM_WORKERS`, `PROFILE_PREFETCH_FACTOR`, and Cell 6 `batch_split` mapping.
- **Checkpoints:** `args.auto_resume=True` (default). **Cell 6** reloads the **latest** `checkpoint-*.pth` under **`CHECKPOINT_DIR`** each time you run it (`resume_path` is cleared so discovery is not stale). Override with env **`SIGNVLM_RESUME_PATH=/path/to/checkpoint-NNNN.pth`**. To **start from step 0**, use **`SIGNVLM_NO_AUTO_RESUME=1`** or `auto_resume=False` in Cell 4.
- **I/O vs GPU:** set **`IO_SPEED_TEST_DUMMY = True`** in the code cell below to use `dummy_dataset` (fake tensors, same shapes) and compare printed step times to real data. A large gap means disk/decode/augment bound, not GPU matmul bound.
- **Future:** with `frames_available=0`, `video_dataset/dataset.py` currently **decodes whole videos** per sample. For long-term speed (especially if you skip staging), plan **partial PyAV decode / seek** so only the needed temporal span is decoded (see comment block in `dataset.py` next to `av.open`).

In [ ]:
# Cell 4: Build label map and TSV lists; create train/val DataLoaders via repo helpers
from pathlib import Path
import argparse
import json
import os

FRAMES_AVAILABLE = 0  # 1 if using pre-extracted frames (see SIGNVLM_TRAINING_NOTES.md)

# True = no real disk reads in train_loader (same tensor shapes); use to compare vs real data I/O.
IO_SPEED_TEST_DUMMY = False


def sorted_label_dirs(root: Path):
    return sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name.casefold())


def build_label_map_from_train(train_root: Path):
    labels = [d.name for d in sorted_label_dirs(train_root)]
    return {name: i for i, name in enumerate(labels)}, labels


def iter_videos(label_dir: Path):
    vids = sorted(label_dir.glob("*.mp4"), key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else p.stem))
    for v in vids:
        yield v


def write_split_tsv(split_root: Path, rel_prefix: Path, name_to_id: dict, out_path: Path):
    n = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for label_dir in sorted_label_dirs(split_root):
            name = label_dir.name
            if name not in name_to_id:
                raise ValueError(f"Unknown label folder {name!r} under {split_root}; not in train label map")
            lid = name_to_id[name]
            for vid in iter_videos(label_dir):
                rel = (rel_prefix / name / vid.name).as_posix()
                f.write(f"{rel}\t{lid}\n")
                n += 1
    return n


def write_flat_unseen_tsv(split_root: Path, name_to_id: dict, out_path: Path):
    n = 0
    vids = sorted(
        [p for p in split_root.iterdir() if p.is_file() and p.suffix.lower() == ".mp4"],
        key=lambda p: p.name.casefold(),
    )
    with open(out_path, "w", encoding="utf-8") as f:
        for vid in vids:
            name = vid.stem
            if name not in name_to_id:
                raise ValueError(
                    f"Unknown label {name!r} from {vid.name} under {split_root}; not in train label map"
                )
            lid = name_to_id[name]
            rel = f"./{vid.name}"
            f.write(f"{rel}\t{lid}\n")
            n += 1
    return n


def build_unseen_tsv(unseen_root: Path, name_to_id: dict, out_path: Path):
    """Flat unseen_data: *.mp4 in root, stem=class. Else: class subfolders like train (write_split_tsv)."""
    if any(p.is_file() and p.suffix.lower() == ".mp4" for p in unseen_root.iterdir()):
        return write_flat_unseen_tsv(unseen_root, name_to_id, out_path)
    return write_split_tsv(unseen_root, Path("."), name_to_id, out_path)


def require_split_dir(path: str, split_label: str):
    if not os.path.isdir(path):
        raise FileNotFoundError(
            f"{split_label} directory missing: {path}\n"
            f"Expected under BASE={BASE} (see Cell 2). Create class subfolders with .mp4 files."
        )


require_split_dir(TRAIN_DIR, "train")
require_split_dir(VAL_DIR, "validation")
require_split_dir(TEST_DIR, "test")

train_root = Path(TRAIN_DIR)
name_to_id, class_names = build_label_map_from_train(train_root)
NUM_CLASSES = len(name_to_id)

label_json = Path(LIST_DIR) / "label_map_auto.json"
with open(label_json, "w", encoding="utf-8") as f:
    json.dump({str(i): class_names[i] for i in range(len(class_names))}, f, ensure_ascii=False, indent=2)
print(f"num_classes={NUM_CLASSES}, wrote {label_json}")

train_tsv = Path(LIST_DIR) / "train.tsv"
val_tsv = Path(LIST_DIR) / "val.tsv"
test_tsv = Path(LIST_DIR) / "test.tsv"
unseen_tsv = Path(LIST_DIR) / "unseen.tsv"


def _count_tsv_lines(p: Path) -> int:
    with open(p, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)


# Keep False when data changes (e.g., after augmentation) so split TSVs are rebuilt.
REUSE_EXISTING_TSV_LISTS = False

_ready_train = train_tsv.is_file() and train_tsv.stat().st_size > 0
_ready_val = val_tsv.is_file() and val_tsv.stat().st_size > 0
_ready_test = test_tsv.is_file() and test_tsv.stat().st_size > 0
_ready_unseen = unseen_tsv.is_file() and unseen_tsv.stat().st_size > 0

if REUSE_EXISTING_TSV_LISTS and _ready_train:
    n_train = _count_tsv_lines(train_tsv)
else:
    n_train = write_split_tsv(train_root, Path("."), name_to_id, train_tsv)

if REUSE_EXISTING_TSV_LISTS and _ready_val:
    n_val = _count_tsv_lines(val_tsv)
else:
    n_val = write_split_tsv(Path(VAL_DIR), Path("."), name_to_id, val_tsv)

if REUSE_EXISTING_TSV_LISTS and _ready_test:
    n_test = _count_tsv_lines(test_tsv)
else:
    n_test = write_split_tsv(Path(TEST_DIR), Path("."), name_to_id, test_tsv)

INCLUDE_UNSEEN = os.path.isdir(UNSEEN_DIR)
if INCLUDE_UNSEEN:
    if REUSE_EXISTING_TSV_LISTS and _ready_unseen:
        n_unseen = _count_tsv_lines(unseen_tsv)
    else:
        n_unseen = build_unseen_tsv(Path(UNSEEN_DIR), name_to_id, unseen_tsv)
else:
    unseen_tsv.write_text("", encoding="utf-8")
    n_unseen = 0
    print(f"INCLUDE_UNSEEN=False: missing {UNSEEN_DIR} - wrote empty {unseen_tsv}")

if REUSE_EXISTING_TSV_LISTS:
    print("Cell 4: reuse enabled (existing split TSVs reused if present).")
else:
    print("Cell 4: reuse disabled -> recreated TSVs for all available datasets.")

print(f"lines: train={n_train} val={n_val} test={n_test} unseen={n_unseen}")

_work = globals().get("WORK_ROOT", "/content")
_in_c = globals().get("IN_COLAB", False)
# Cache-on-first-use from Drive to local SSD (no bulk copy). Set LOCAL_DATA_CACHE = "" to disable.
LOCAL_DATA_CACHE = ""
if _in_c:
    LOCAL_DATA_CACHE = os.path.join(_work, "signvlm_data_cache")
    os.makedirs(LOCAL_DATA_CACHE, exist_ok=True)
_ncpu = os.cpu_count() or 4

# ----------------------------
# Hardcoded training profiles
# ----------------------------
TRAIN_SPEED_PROFILE = "max_speed"  # change manually: "fast_stable" or "max_speed"
if TRAIN_SPEED_PROFILE == "max_speed":
    PROFILE_BATCH_SIZE = 20
    PROFILE_NUM_WORKERS = min(16, max(6, (2 * _ncpu) // 3))
    PROFILE_PREFETCH_FACTOR = 4
elif TRAIN_SPEED_PROFILE == "fast_stable":
    PROFILE_BATCH_SIZE = 12
    PROFILE_NUM_WORKERS = min(4, max(2, (_ncpu // 4) or 2))
    PROFILE_PREFETCH_FACTOR = 2
else:
    raise ValueError(f"Unknown TRAIN_SPEED_PROFILE={TRAIN_SPEED_PROFILE!r}")

_batch_size = PROFILE_BATCH_SIZE
_num_workers = PROFILE_NUM_WORKERS
_prefetch_factor = PROFILE_PREFETCH_FACTOR
_high = TRAIN_SPEED_PROFILE == "max_speed"

# Namespace with fields expected by video_dataset + main (PSL-style hyperparameters from SIGNVLM_TRAINING_NOTES)
_no_auto_resume = os.environ.get("SIGNVLM_NO_AUTO_RESUME", "").strip().lower() in ("1", "true", "yes", "on")
args = argparse.Namespace(
    frames_available=FRAMES_AVAILABLE,
    train_list_path=str(train_tsv),
    val_list_path=str(val_tsv),
    train_data_root=TRAIN_DIR,
    val_data_root=VAL_DIR,
    data_root="",
    local_data_cache=LOCAL_DATA_CACHE,
    batch_size=12,
    n_shots=-1,
    num_spatial_views=1,
    num_temporal_views=3,
    num_frames=24,
    sampling_rate=4,
    tsn_sampling=False,
    spatial_size=224,
    mean=[0.48145466, 0.4578275, 0.40821073],
    std=[0.26862954, 0.26130258, 0.27577711],
    num_workers=_num_workers,
    pin_memory=True,
    persistent_workers=_num_workers > 0,
    prefetch_factor=_prefetch_factor,
    dummy_dataset=IO_SPEED_TEST_DUMMY,
    auto_augment="rand-m7-n4-mstd0.5-inc1",
    interpolation="bicubic",
    mirror=True,
    num_steps=10000,
    checkpoint_dir=CHECKPOINT_DIR,
    auto_resume=not _no_auto_resume,
    resume_path=None,
    pretrain=None,
)

train_loader = video_dataset.create_train_loader(args, resume_step=0)
val_loader = video_dataset.create_val_loader(args)
unseen_loader = None
_unseen_tsv = Path(unseen_tsv)
if INCLUDE_UNSEEN and _unseen_tsv.is_file() and _unseen_tsv.stat().st_size > 0:
    _unseen_ns = argparse.Namespace(**vars(args))
    _unseen_ns.val_list_path = str(unseen_tsv)
    _unseen_ns.val_data_root = UNSEEN_DIR
    _unseen_ns.num_spatial_views = 1
    _unseen_ns.num_temporal_views = 3
    _unseen_ns.dummy_dataset = False
    _unseen_ns.auto_augment = None
    _unseen_ns.mirror = False
    unseen_loader = video_dataset.create_val_loader(_unseen_ns)
    print("unseen samples:", len(unseen_loader.dataset), "(source:", UNSEEN_DIR, ")")
else:
    print("unseen loader: (off) missing folder or empty unseen.tsv")
print(
    "train_loader batches (preview resume_step=0):", len(train_loader),
    "| num_steps:", args.num_steps,
    "| Cell 6 loads latest checkpoint if auto_resume; Cell 7 rebuilds loader with resume_step.",
)
print("val samples:", len(val_loader.dataset))
print(
    "DataLoader: num_workers=", args.num_workers,
    "prefetch_factor=", args.prefetch_factor,
    "batch_size=", args.batch_size,
    "dummy_dataset=", args.dummy_dataset,
    "local_data_cache=", getattr(args, "local_data_cache", "") or "(off)",
    "profile=", TRAIN_SPEED_PROFILE,
    "signvlm_high_ram=", _high,
)


def make_eval_split_loader(tsv, root):
    # Eval-style loader (multi-view), same settings as val - train/test/unseen metrics.
    ns = argparse.Namespace(
        frames_available=args.frames_available,
        val_list_path=str(tsv),
        val_data_root=root,
        data_root="",
        local_data_cache=getattr(args, "local_data_cache", None) or "",
        batch_size=args.batch_size,
        n_shots=-1,
        num_spatial_views=1,
        num_temporal_views=3,
        num_frames=args.num_frames,
        sampling_rate=args.sampling_rate,
        tsn_sampling=False,
        spatial_size=args.spatial_size,
        mean=args.mean,
        std=args.std,
        num_workers=args.num_workers,
        pin_memory=getattr(args, "pin_memory", True),
        persistent_workers=getattr(args, "persistent_workers", args.num_workers > 0),
        prefetch_factor=getattr(args, "prefetch_factor", 3),
        dummy_dataset=False,
    )
    return video_dataset.create_val_loader(ns)


num_classes=104, wrote /content/drive/MyDrive/FYP_DATA/signVLM_lists/label_map_auto.json
INCLUDE_UNSEEN=False: missing /content/drive/MyDrive/FYP_DATA/Unseen_data - wrote empty /content/drive/MyDrive/FYP_DATA/signVLM_lists/unseen.tsv
lines: train=1556 val=312 test=101 unseen=0
1556
312
train_loader steps: 10000 expected: 10000
val samples: 312
DataLoader: num_workers= 2 prefetch_factor= 2 dummy_dataset= False local_data_cache= /content/signvlm_data_cache


## Cell 5: Model — `EVLTransformer` (signVLM)

Matches **PSL** script settings: **ViT-L/14-lnpre**, decoder **4×1024**, **16 heads**, temporal conv / pos / cross-attention enabled (`docs/signVLMPipeline.md`). Backbone defaults to **frozen fp16** unless you set `finetune_backbone=True`.

In [8]:
# Cell 5: EVLTransformer (ViT-L/14 + decoder)
finetune_backbone = False  # set True for end-to-end (see docs/signVLMPipeline.md)
fp16 = True

model = EVLTransformer(
    backbone_name="ViT-L/14-lnpre",
    backbone_type="clip",
    backbone_path=BACKBONE_PATH,
    backbone_mode="finetune" if finetune_backbone else ("freeze_fp16" if fp16 else "freeze_fp32"),
    decoder_num_layers=4,
    decoder_qkv_dim=1024,
    decoder_num_heads=16,
    decoder_mlp_factor=4.0,
    num_classes=NUM_CLASSES,
    enable_temporal_conv=True,
    enable_temporal_pos_embed=True,
    enable_temporal_cross_attention=True,
    cls_dropout=0.5,
    decoder_mlp_dropout=0.5,
    num_frames=args.num_frames,
)
model.cuda()
print(model.__class__.__name__, "num_classes:", NUM_CLASSES)

---- backbone_path:  ViT-L/14-lnpre
EVLTransformer num_classes: 104


## Cell 6: Training configuration

- **Optimizer:** AdamW, `lr=4e-5`, `weight_decay=0.05` (`main.py` / notes).
- **Scheduler:** `CosineAnnealingLR(T_max=num_steps)`.
- **Loss:** `CrossEntropyLoss`.
- **AMP:** `GradScaler` + `autocast` (disable with `fp16=False`).
- **`batch_split`:** controlled by hardcoded `TRAIN_SPEED_PROFILE` from Cell 4.
  - `"fast_stable"` -> `batch_split=2`
  - `"max_speed"` -> `batch_split=1`
  Edit manually if needed. OOM: use `batch_split=2` or lower batch size in Cell 4.
- **Resume:** `resume_from_checkpoint(...)` uses **`args.auto_resume`** and **`CHECKPOINT_DIR`** (or **`SIGNVLM_RESUME_PATH`**). **Cell 6** clears `resume_path` before discovery so a re-run still picks the newest file. **Cell 7** rebuilds `train_loader` with **`resume_step`** from `next_step` in the checkpoint.

In [ ]:
# Cell 6: Optimizer, scheduler, loss, AMP
import os

lr = 4e-5
weight_decay = 0.05

# Use the hardcoded profile from Cell 4
if TRAIN_SPEED_PROFILE == "max_speed":
    batch_split = 1
elif TRAIN_SPEED_PROFILE == "fast_stable":
    batch_split = 2
else:
    raise ValueError(f"Unknown TRAIN_SPEED_PROFILE={TRAIN_SPEED_PROFILE!r}")
if args.batch_size % batch_split != 0:
    raise ValueError(f"batch_size {args.batch_size} must be divisible by batch_split {batch_split}")
num_steps = args.num_steps
save_freq = 2500
eval_freq = 2500
# Enable confusion + F1 during training eval. Set 0 only if you want to disable it.
confusion_eval_freq = eval_freq

print_freq = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)
loss_scaler = torch.amp.GradScaler("cuda", enabled=fp16)
criterion = torch.nn.CrossEntropyLoss()

args.save_freq = save_freq
args.eval_freq = eval_freq
args.print_freq = print_freq
args.confusion_eval_freq = confusion_eval_freq

args.batch_split = batch_split
args.fp16 = fp16
args.finetune_backbone = finetune_backbone

# Resume: re-scan CHECKPOINT_DIR each run when auto_resume (stale args.resume_path would skip discovery).
_resume_explicit = os.environ.get("SIGNVLM_RESUME_PATH", "").strip()
if _resume_explicit:
    args.resume_path = os.path.abspath(_resume_explicit)
elif getattr(args, "auto_resume", False):
    args.resume_path = None
else:
    args.resume_path = None  # fresh run; do not reuse a stale path from an old session

resume_step = checkpoint.resume_from_checkpoint(model, optimizer, lr_sched, loss_scaler, args)
print("resume_step:", resume_step, "batch_split:", batch_split)


Not resuming from a checkpoint.
resume_step: 0


## Cell 7: Training and validation loop

Mirrors `main.py`: micro-batching with `batch_split`, `GradScaler`, cosine LR step each iteration, periodic **`evaluate()`** (top-1 / top-5) on `val_loader`, and **validation confusion matrix + F1** when **`confusion_eval_freq == eval_freq`** (default; set `confusion_eval_freq = 0` in Cell 6 to skip). Training-time eval runs on **validation only** (no train-split eval in this cell). Checkpoints go to **Google Drive** (`CHECKPOINT_DIR`).

CLI alternative: `python main.py --train_list_path ... --val_list_path ... --checkpoint_dir ...` (see `scripts/train_psl_vitl14_16f_dec4x1024_1shot.sh`).

In [10]:
# Cell 7: Training + validation (same structure as main.py)
from datetime import datetime

import main as signvlm_main

train_loader = video_dataset.create_train_loader(args, resume_step=resume_step)
assert len(train_loader) == args.num_steps - resume_step

model.train()
train_st = datetime.now()
batch_st = datetime.now()

for i, (data, labels) in enumerate(train_loader, resume_step):
    data = data.cuda(non_blocking=True)
    labels = labels.cuda(non_blocking=True)
    data_ed = datetime.now()
    optimizer.zero_grad(set_to_none=True)

    assert data.size(0) % args.batch_split == 0
    split_size = data.size(0) // args.batch_split
    hit1, hit5, loss_value = 0, 0, 0.0

    for j in range(args.batch_split):
        sl = slice(split_size * j, split_size * (j + 1))
        data_slice, labels_slice = data[sl], labels[sl]
        with torch.amp.autocast("cuda", enabled=args.fp16):
            logits = model(data_slice)
            loss = criterion(logits, labels_slice)
        if labels.dtype == torch.long:
            hit1 += (logits.topk(1, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
            hit5 += (logits.topk(5, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
        loss_value += loss.item() / args.batch_split
        loss_scaler.scale(loss / args.batch_split).backward()

    loss_scaler.step(optimizer)
    loss_scaler.update()
    lr_sched.step()
    batch_ed = datetime.now()
    data_wait_s = (data_ed - batch_st).total_seconds()
    compute_s = (batch_ed - data_ed).total_seconds()
    _done = i - resume_step + 1
    _left = args.num_steps - i - 1
    eta = ((batch_ed - train_st) / _done) * _left if _done > 0 else 0

    if i % args.print_freq == 0:
        sync_tensor = torch.tensor([loss_value, hit1 / data.size(0), hit5 / data.size(0)], device="cuda")
        dist.all_reduce(sync_tensor)
        sync_tensor = sync_tensor.cpu() / dist.get_world_size()
        loss_value, acc1, acc5 = sync_tensor.tolist()
        print(
            f"Step {i}/{args.num_steps} batch {(batch_ed - batch_st).total_seconds():.3f}s "
            f"wait {data_wait_s:.3f}s compute {compute_s:.3f}s "
            f"lr {optimizer.param_groups[0]['lr']:.6f} loss {loss_value:.6f} "
            f"acc1 {acc1 * 100:.2f}% acc5 {acc5 * 100:.2f}% eta {eta}"
        )

    if (i + 1) % args.save_freq == 0:
        checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, i + 1, args)

    if (i + 1) % args.eval_freq == 0:
        print("Eval at step", i + 1)
        model.eval()
        signvlm_main.evaluate(model, val_loader)
        if args.confusion_eval_freq > 0 and (i + 1) % args.confusion_eval_freq == 0:
            yt, yp = collect_predictions(model, val_loader)
            print_confusion_and_f1(yt, yp, NUM_CLASSES, class_names, "validation", save_dir=LIST_DIR)
        model.train()

    batch_st = datetime.now()

checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, num_steps, args)
print("Training done:", datetime.now() - train_st)
print("Skipped train-split eval in Cell 7 (validation-only eval to reduce crash risk).")

1556
Step 0 batch 61.785s lr 0.000040 loss 5.119141 acc1 0.00% acc5 0.00%
Step 10 batch 23.401s lr 0.000040 loss 5.046387 acc1 0.00% acc5 0.00%


KeyboardInterrupt: 

## Cell 8: Evaluation — train, validation, test, unseen

Loads the **latest** `checkpoint-*.pth` from `CHECKPOINT_DIR` (or set `EVAL_CKPT`). Splits run **strictly in order** — **train → validation → test → unseen** — **one split at a time**: each DataLoader is created, evaluated, confusion metrics printed, then **released** before the next split (no parallel loaders; videos are still read lazily per batch).

- **Toggles:** **`CELL8_RUN_TRAIN`, `CELL8_RUN_VAL`, `CELL8_RUN_TEST` default `True`**. **Unseen** needs **`CELL8_RUN_UNSEEN=True`** and `UNSEEN_DIR` on Drive. If `unseen.tsv` is missing/empty at eval time, Cell 8 auto-refreshes it from `UNSEEN_DIR` (same label map as train) and then runs unseen eval when rows exist. **`INCLUDE_UNSEEN` is re-checked in this cell** from `os.path.isdir(UNSEEN_DIR)` so eval is not blocked if Cell 4 ran before `Unseen_data` existed.
- **`CELL8_EVAL_NUM_WORKERS`:** default **`None`** (uses `args.num_workers`, same as training settings). Set an int to override eval workers.
- **`CELL8_MAX_SAMPLES`:** default **`None`** — each split uses all items (set an int only for quick smoke tests).
- **`CELL8_FAST_CONFUSION_PRINT`:** default **`True`** — skips `classification_report` and full confusion-table printing in Cell 8 (still saves full confusion `.npy` and prints a tiny top-left block). Set **`False`** for full verbose output.
- **Requires:** Cells 2–4, 3 (metrics helpers), **Cell 5** (`model`, `args`, paths).

In [ ]:
# Cell 8: Eval train → val → test → unseen (one split at a time; release before next)
import gc
import os
import argparse
from datetime import datetime
from pathlib import Path

import torch
import torch.distributed as dist

CELL8_RUN_TRAIN = True  # run train split first (order: train -> val -> test -> unseen)
CELL8_RUN_VAL = True
CELL8_RUN_TEST = True
CELL8_RUN_UNSEEN = True  # still requires INCLUDE_UNSEEN, folder, and non-empty unseen.tsv
# None = use args.num_workers; int = override eval workers.
CELL8_EVAL_NUM_WORKERS = None
# Per-label cap used only for train split confusion/eval in Cell 8.
CELL8_TRAIN_SAMPLES_PER_LABEL = 15
# True = faster Cell 8 logs (skip classification_report + full matrix print; still save .npy)
CELL8_FAST_CONFUSION_PRINT = True


def latest_checkpoint(ckpt_dir):
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint-") and f.endswith(".pth")]
    if not files:
        return None
    best = max(files, key=lambda f: int(f.replace("checkpoint-", "").replace(".pth", "")))
    return os.path.join(ckpt_dir, best)


def _cell8_nw():
    if CELL8_EVAL_NUM_WORKERS is not None:
        return int(CELL8_EVAL_NUM_WORKERS)
    return int(args.num_workers)


def _cell8_eval_ns(val_list_path, val_data_root):
    nw = _cell8_nw()
    return argparse.Namespace(
        frames_available=args.frames_available,
        val_list_path=str(val_list_path),
        val_data_root=val_data_root,
        data_root="",
        local_data_cache=getattr(args, "local_data_cache", None) or "",
        batch_size=args.batch_size,
        n_shots=-1,
        num_spatial_views=1,
        num_temporal_views=3,
        num_frames=args.num_frames,
        sampling_rate=args.sampling_rate,
        tsn_sampling=False,
        spatial_size=args.spatial_size,
        mean=args.mean,
        std=args.std,
        num_workers=nw,
        pin_memory=getattr(args, "pin_memory", True) and nw > 0,
        persistent_workers=nw > 0,
        prefetch_factor=getattr(args, "prefetch_factor", 3) if nw > 0 else None,
        dummy_dataset=False,
        auto_augment=None,
        interpolation="bicubic",
        mirror=False,
        checkpoint_dir=None,
        auto_resume=False,
        resume_path=None,
        pretrain=None,
    )


def _cell8_subset_indices_per_label(dataset, samples_per_label):
    if samples_per_label is None:
        return list(range(len(dataset)))
    labels = []
    if hasattr(dataset, "data_list"):
        for line in dataset.data_list:
            try:
                labels.append(int(str(line).strip().split("\t")[1]))
            except Exception:
                labels.append(None)
    else:
        return list(range(len(dataset)))

    picked = []
    counts = {}
    for idx, lbl in enumerate(labels):
        if lbl is None:
            continue
        c = counts.get(lbl, 0)
        if c < samples_per_label:
            picked.append(idx)
            counts[lbl] = c + 1
    return picked


def _cell8_create_eval_loader(val_list_path, val_data_root, samples_per_label=None):
    ns = _cell8_eval_ns(val_list_path, val_data_root)
    ds = video_dataset.dataloader.create_val_dataset(ns)
    if samples_per_label is not None:
        keep = _cell8_subset_indices_per_label(ds, samples_per_label)
        if len(keep) < len(ds):
            ds = torch.utils.data.Subset(ds, keep)

    rank, world_size = (0, 1) if not dist.is_initialized() else (dist.get_rank(), dist.get_world_size())
    sampler = list(range(rank, len(ds), world_size))
    kwargs = {
        "num_workers": int(getattr(ns, "num_workers", 0)),
        "pin_memory": bool(getattr(ns, "pin_memory", True)),
    }
    if kwargs["num_workers"] > 0:
        kwargs["persistent_workers"] = bool(getattr(ns, "persistent_workers", True))
        kwargs["prefetch_factor"] = int(getattr(ns, "prefetch_factor", 2))
    return torch.utils.data.DataLoader(
        ds,
        sampler=sampler,
        batch_size=1,
        **kwargs,
    )


def _cell8_evaluate_named(model, loader, split_name):
    tot, hit1, hit5 = 0, 0, 0
    eval_st = datetime.now()
    for data, labels in loader:
        data = data.cuda(non_blocking=True)
        labels = labels.cuda(non_blocking=True)
        assert data.size(0) == 1
        if data.ndim == 6:
            data = data[0]

        with torch.no_grad():
            logits = model(data)
            scores = logits.softmax(dim=-1).mean(dim=0)

        tot += 1
        hit1 += (scores.topk(1)[1] == labels).sum().item()
        hit5 += (scores.topk(5)[1] == labels).sum().item()

        if tot % 20 == 0:
            print(
                f"[Evaluation:{split_name}] num_samples: {tot}  "
                f"ETA: {(datetime.now() - eval_st) / tot * (len(loader) - tot)}  "
                f"cumulative_acc1: {hit1 / tot * 100.:.2f}%  "
                f"cumulative_acc5: {hit5 / tot * 100.:.2f}%"
            )

    sync_tensor = torch.LongTensor([tot, hit1, hit5]).cuda()
    dist.all_reduce(sync_tensor)
    tot, hit1, hit5 = sync_tensor.cpu().tolist()
    print(f"Total {split_name} time: {datetime.now() - eval_st}, Average {split_name} time:{(datetime.now() - eval_st) / len(loader)}")
    print(f"Accuracy on {split_name} set: top1={hit1 / tot * 100:.2f}%, top5={hit5 / tot * 100:.2f}%")


def _cell8_run_split(title, val_list_path, val_data_root):
    """Single split: build loader, evaluate, confusion, then drop loader and arrays."""
    print(f"=== {title} ===")
    split_samples_per_label = CELL8_TRAIN_SAMPLES_PER_LABEL if title == "train" else None
    if split_samples_per_label is not None:
        print(f"  (train-only cap: up to {split_samples_per_label} samples per label)")
    loader = None
    try:
        loader = _cell8_create_eval_loader(
            val_list_path,
            val_data_root,
            samples_per_label=split_samples_per_label,
        )
        print(f"  samples this run: {len(loader.dataset)}")
        _cell8_evaluate_named(model, loader, title)
        yt, yp = collect_predictions(model, loader)
        _no_matrix_titles = {"train", "validation"}
        _print_matrix = (not CELL8_FAST_CONFUSION_PRINT) and (title not in _no_matrix_titles)
        print_confusion_and_f1(
            yt,
            yp,
            NUM_CLASSES,
            class_names,
            title,
            save_dir=LIST_DIR,
            print_report=not CELL8_FAST_CONFUSION_PRINT,
            print_matrix=_print_matrix,
        )
    finally:
        if loader is not None:
            _ds = getattr(loader, "dataset", None)
            del loader
            if _ds is not None:
                del _ds
        try:
            del yt, yp
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"--- done: {title} (released loader) ---\n")


EVAL_CKPT = latest_checkpoint(CHECKPOINT_DIR) or os.path.join(CHECKPOINT_DIR, "checkpoint-10000.pth")

model.eval()
ckpt = torch.load(EVAL_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
del ckpt
gc.collect()
print("Loaded:", EVAL_CKPT)

# Re-sync unseen flag from Drive at eval time (Cell 4 may be stale if Unseen_data was added later)
INCLUDE_UNSEEN = bool(os.path.isdir(UNSEEN_DIR))
print(f"Eval: INCLUDE_UNSEEN={INCLUDE_UNSEEN} UNSEEN_DIR={UNSEEN_DIR}")

if CELL8_RUN_TRAIN:
    _cell8_run_split("train", train_tsv, TRAIN_DIR)
else:
    print("Skipping train eval (CELL8_RUN_TRAIN=False).")

if CELL8_RUN_VAL:
    _cell8_run_split("validation", val_tsv, VAL_DIR)
else:
    print("Skipping validation eval (CELL8_RUN_VAL=False).")

if CELL8_RUN_TEST:
    _cell8_run_split("test", test_tsv, TEST_DIR)
else:
    print("Skipping test eval (CELL8_RUN_TEST=False).")

_run_unseen = CELL8_RUN_UNSEEN and INCLUDE_UNSEEN
_ts = Path(unseen_tsv)
if _run_unseen and (not _ts.is_file() or _ts.stat().st_size == 0):
    try:
        _n_unseen = build_unseen_tsv(Path(UNSEEN_DIR), name_to_id, _ts)
        print(f"Refreshed unseen.tsv for Cell 8: {_n_unseen} rows")
    except Exception as _e:
        print(f"Could not refresh unseen.tsv in Cell 8: {_e}")

if _run_unseen and _ts.is_file() and _ts.stat().st_size > 0:
    _cell8_run_split("unseen", unseen_tsv, UNSEEN_DIR)
else:
    print(
        "Skipping Unseen_data (need CELL8_RUN_UNSEEN=True and Unseen_data folder on Drive; unseen.tsv is auto-refreshed in Cell 8 when possible)."
    )
